本 Notebook 使用离线构造数据。固定申报事实与图表属于历史教学材料；新计算不复现市场策略。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch11_returns_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第11章：收益、复利与路径风险

默认可执行格仅计算构造路径。French NoDur与同月RF的历史计算保留为围栏源代码，不在默认路线执行，市场文件不随包。月份是收益区间标签，不等于信号发布或成交时间；当前修订快照的描述数字不证明历史可交易策略。

In [ ]:
from pathlib import Path
import json
import numpy as np
from quant_course.market_data import load_monthly_snapshot
from quant_course.performance import (wealth_index, geometric_mean_return, cagr,
    drawdown_path, annualized_volatility, sharpe_ratio)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "请从课程根目录或 notebooks 目录运行"


## 构造案例：百分数相加会丢掉复利

100先涨50%再跌50%，财富剩75。算术平均为零，几何平均为负；观察初始财富在回撤中的作用。

In [ ]:
returns = np.array([0.5, -0.5])
wealth = wealth_index(returns, initial_capital=100)
assert np.allclose(wealth, [100, 150, 75])
print({"wealth": wealth, "arithmetic_mean": returns.mean(),
       "geometric_mean": geometric_mean_return(returns),
       "drawdown": drawdown_path(wealth)})


## 同终值不同风险

下面两条路径使用相同三期收益。每期设为一年，CAGR相同，最大回撤分别为52%和40%。

In [ ]:
for sequence in ([0.5, -0.4, -0.2], [-0.4, 0.5, -0.2]):
    values = wealth_index(np.array(sequence), 100)
    print(values, cagr(100, float(values[-1]), 3), -drawdown_path(values).min())


## 历史快照示例（默认禁用）

围栏代码展示原字节核验后对齐2010-01至2019-12共120个月的办法。处理收益已为小数，不再除100；不补缺月或把RF填零。它是历史源说明，不是本次默认输出。

历史French分支默认跳过：French 数据权利和执行授权尚未确立，无市场文件随包。以下围栏仅保留历史源说明，不执行；其中QA路径不是便携包运行依赖。

```python
industry = json.loads((ROOT / "data/manifests/french_industries_2026-10-05.json").read_text())
factor = json.loads((ROOT / "data/manifests/french_factors_2026-10-05.json").read_text())["datasets"]["ff3"]
asset = load_monthly_snapshot(ROOT / industry["processed_path"], industry["processed_sha256"], ["NoDur"])
rf = load_monthly_snapshot(ROOT / factor["processed_path"], factor["processed_sha256"], ["RF"])
sample = asset.loc["2010-01":"2019-12"].join(rf, how="left", validate="one_to_one")
assert len(sample) == 120 and not sample.isna().any().any()
assert (sample.to_numpy() > -1).all()
r = sample["NoDur"].to_numpy()
risk_free = sample["RF"].to_numpy()
values = wealth_index(r)
metrics = {"final_wealth": float(values[-1]), "CAGR": cagr(1, float(values[-1]), 10),
    "volatility_sqrt12_convention": annualized_volatility(r, 12),
    "Sharpe_sqrt12_convention": sharpe_ratio(r, risk_free, 12),
    "maximum_drawdown_positive_loss": -float(drawdown_path(values).min())}
metrics

```

## 可选历史图的读法

四图分别为初始1的财富（非价格）、小数月收益分布、过去12月样本波动率乘√12，以及相对既往峰值的负回撤。年化是报告约定。绘图代码同样处于围栏中，默认不执行，也不新写图。

历史French分支默认跳过：French 数据权利和执行授权尚未确立，无市场文件随包。以下围栏仅保留历史源说明，不执行；其中QA路径不是便携包运行依赖。

```python
import runpy
from IPython.display import Image, display
plot_helpers = runpy.run_path(str(ROOT / "code/ch11_returns.py"), run_name="plot_helpers")
figure = RUN_DIR / "figures/a3/ch11_returns_dashboard.png"
plot_helpers["make_figure"](sample, figure)
display(Image(filename=str(figure)))

```

## 自检与研究边界

末尾围栏保留旧QA指标文件引用，仅属历史源说明，不是便携包运行依赖。此处没有交易规则、费用、入场时点或样本外预测，不能称Alpha回测。真实行业组合快照也不是逐股PIT数据库。

历史French分支默认跳过：French 数据权利和执行授权尚未确立，无市场文件随包。以下围栏仅保留历史源说明，不执行；其中QA路径不是便携包运行依赖。

```python
recorded = json.loads((ROOT / "qa/a3/ch11_actual_metrics.json").read_text())
assert np.isclose(metrics["final_wealth"], recorded["wealth_final"])
assert np.isclose(metrics["CAGR"], recorded["cagr_month_period_convention"])
assert np.isclose(metrics["Sharpe_sqrt12_convention"], recorded["sharpe_sqrt12_convention"])
print("同一快照指标一致；不构成策略收益证据。")

```